# ResNet50 for Image Classification

This notebook trains a ResNet50 (Transfer Learning) model on the image dataset.

In [ ]:
import os
import cv2
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, applications
from sklearn.model_selection import train_test_split

IMG_SIZE = (64, 64)
DATA_DIR = "data"
RESULTS_FILE = "results/model_comparison.csv"

In [ ]:
# --- Data Loading Helper ---
def load_data():
    if not os.path.exists(DATA_DIR) or not os.listdir(DATA_DIR):
        print("Generating synthetic data...")
        X, y = [], []
        for i in range(200):
            label = i % 3
            img = np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8)
            if label == 0: img[:,:,0] += 50
            elif label == 1: img[:,:,1] += 50
            else: img[:,:,2] += 50
            X.append(img)
            y.append(label)
        return np.array(X), np.array(y), 3
    
    print("Loading from folders...")
    X, y = [], []
    classes = sorted(os.listdir(DATA_DIR))
    num_classes = len(classes)
    for i, cls in enumerate(classes):
        path = os.path.join(DATA_DIR, cls)
        if not os.path.isdir(path): continue
        for f in os.listdir(path):
            try:
                img = cv2.imread(os.path.join(path, f))
                if img is not None:
                    X.append(cv2.resize(img, IMG_SIZE))
                    y.append(i)
            except: pass
    return np.array(X), np.array(y), num_classes

X, y, NUM_CLASSES = load_data()
# Preprocess for DL
X = X.astype('float32') / 255.0

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print("Training ResNet50...")
base_model = applications.ResNet50(weights='imagenet', include_top=False, input_shape=(64, 64, 3))
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(NUM_CLASSES, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

start = time.time()
model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)
train_time = time.time() - start

start = time.time()
loss, acc = model.evaluate(X_test, y_test, verbose=0)
inf_time = (time.time() - start) / len(X_test)

params = model.count_params()
print(f"Done. Acc: {acc:.4f}, Time: {train_time:.4f}s")

In [ ]:
# Save Results
os.makedirs("results", exist_ok=True)
new_row = {
    "Model": "ResNet50",
    "Type": "DL",
    "Accuracy": acc,
    "Training Time (s)": train_time,
    "Inference Time (s/sample)": inf_time,
    "Parameters": params
}

if os.path.exists(RESULTS_FILE):
    df = pd.read_csv(RESULTS_FILE)
    df = df[df['Model'] != 'ResNet50']
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
else:
    df = pd.DataFrame([new_row])
    
df.to_csv(RESULTS_FILE, index=False)
print("Saved to results/model_comparison.csv")